# Demo · Asesor de viajes multi-agente (Pre-entrega 6)

Este notebook muestra el **flujo de delegación** del orquestador jerárquico:

1. El grafo y su diagrama.
2. La pregunta de demo en **modo mock** (sin claves): cada decisión del supervisor, qué hizo cada especialista y la respuesta final.
3. La misma pregunta **en vivo** (Claude + Tavily), si hay claves en el `.env`.
4. La segunda vez que se pregunta lo mismo: los datos salen de la **memoria** (ChromaDB) sin llamar a Tavily.
5. El **freno de pasos** contra el "supervisor infinito".

> Para reejecutarlo: `.venv\Scripts\python.exe -m jupyter nbconvert --to notebook --execute --inplace demo.ipynb`

In [1]:
import logging
import warnings
from datetime import date

from IPython.display import Markdown, display

import config
from graph import construir_grafo, consultar, crear_componentes, diagrama_mermaid

warnings.filterwarnings("ignore")
config.configurar_logs(logging.WARNING)
HAY_CLAVES = bool(config.ANTHROPIC_API_KEY and config.TAVILY_API_KEY)
DEMO = ("Quiero ir 7 días a Bariloche en enero, somos 2 personas, salimos desde Mendoza y tenemos "
        "800.000 pesos. ¿Me armás el viaje y me decís si nos alcanza?")
print("Fecha de referencia:", config.FECHA_REFERENCIA, "| claves en el .env:", "sí" if HAY_CLAVES else "no (solo mock)")

Fecha de referencia: 2026-11-10 | claves en el .env: sí


## 1. El grafo

Topología **jerárquica**: el supervisor está en el centro y decide, con `add_conditional_edges` y una función que devuelve `Literal["investigador", "analista", "sintesis"]`, quién trabaja ahora. Los especialistas siempre le devuelven el control. Este es el diagrama que genera LangGraph (`graph.get_graph().draw_mermaid()`):

In [2]:
componentes_mock = crear_componentes(modo_mock=True)
grafo_mock = construir_grafo(componentes_mock)
print(diagrama_mermaid(grafo_mock))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	investigador(investigador)
	analista(analista)
	sintesis(sintesis)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analista --> supervisor;
	investigador --> supervisor;
	supervisor -.-> analista;
	supervisor -.-> investigador;
	supervisor -.-> sintesis;
	sintesis --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. La pregunta de demo en modo mock (sin claves)

En modo mock el LLM y Tavily están simulados, pero **el grafo, las herramientas del analista, la memoria ChromaDB y las validaciones son los reales**. Esta función muestra el recorrido paso a paso a partir de la traza:

In [3]:
def mostrar_recorrido(traza):
    print("RUTA:", traza.texto_ruta())
    print()
    for e in traza.eventos:
        d = e.detalle
        if e.nodo == "supervisor":
            if "pedido_extraido" in d:
                print("SUPERVISOR · pedido extraído:", d["pedido_extraido"])
            freno = "  (FORZADA POR EL FRENO)" if d.get("forzada") else ""
            print(f"SUPERVISOR · decisión {d['paso']} → {d['siguiente']}{freno}")
            print("   evaluación (rúbrica):", d["evaluacion"][:400])
            if d.get("temas"):
                print("   temas pedidos:", ", ".join(d["temas"]))
        elif e.nodo in ("investigador", "analista"):
            print(f"{e.nodo.upper()} · herramientas: {', '.join(d['herramientas']) or 'ninguna'} "
                  f"· llamadas: {len(d['llamadas'])}")
            for c in d.get("cobertura", []):
                print(f"   cobertura {c['tema']}: {c['fuentes']} fuentes ({c['memoria']} memoria / {c['web']} web)"
                      f"{' · con precios' if c['con_precios'] else ''}")
            print("   chequeo por código:", "; ".join(d["observaciones"]) or "sin observaciones")
        else:
            print("SÍNTESIS · respuesta final lista")
    m = traza.metricas
    print()
    print(f"Decisiones: {m['pasos_supervisor']} | validación: {m['validacion']} | ¿entra?: {m['presupuesto_entra']} "
          f"| datos: {traza.texto_origenes()} | {traza.etiqueta_busquedas()}: {m['busquedas_web']} | {traza.duracion_s} s")


traza_mock = await consultar(grafo_mock, DEMO, escenario="notebook_demo", modo_mock=True)
mostrar_recorrido(traza_mock)

RUTA: supervisor → investigador → supervisor → analista → supervisor → sintesis

SUPERVISOR · pedido extraído: {'origen': 'Mendoza', 'destino': 'Bariloche', 'fecha_ida': '2027-01-10', 'fecha_vuelta': '2027-01-16', 'dias': 7, 'personas': 2, 'presupuesto': 800000.0, 'moneda': 'ARS', 'supuestos': ['enero sin día exacto: se asume del 10/01/2027 al 16/01/2027']}
SUPERVISOR · decisión 1 → investigador
   evaluación (rúbrica): R1: todavía no hay datos. Empieza el investigador.
   temas pedidos: pasajes, alojamiento, clima, actividades, cambio
INVESTIGADOR · herramientas: buscar_en_memoria, buscar_en_web · llamadas: 10
   cobertura pasajes: 2 fuentes (0 memoria / 2 web) · con precios
   cobertura alojamiento: 1 fuentes (0 memoria / 1 web) · con precios
   cobertura clima: 1 fuentes (0 memoria / 1 web)
   cobertura actividades: 1 fuentes (0 memoria / 1 web) · con precios
   cobertura cambio: 1 fuentes (0 memoria / 1 web) · con precios
   chequeo por código: sin observaciones
SUPERVISOR · decisi

La respuesta final (con el itinerario, el presupuesto desglosado y las fuentes armadas por código):

In [4]:
display(Markdown(traza_mock.respuesta_final))

## Itinerario
- Día 1 (10/01): viaje desde Mendoza y llegada a Bariloche
- Día 2 (11/01): Circuito Chico en bici
- Día 3 (12/01): trekking al Refugio Frey
- Día 4 (13/01): playas del lago Gutiérrez
- Día 5 (14/01): excursión a Isla Victoria y Bosque de Arrayanes
- Día 6 (15/01): día libre: descanso y paseo por el centro
- Día 7 (16/01): regreso a Mendoza

Clima esperado: Enero es pleno verano en Bariloche: máximas promedio de 22 °C y mínimas de 7 °C, con días largos (anochece cerca de las 21:30). Llueve poco (unos 4 días al mes), pero hay viento y las noches son frescas: conviene llevar abrigo liviano y protector solar.

## Presupuesto
PRESUPUESTO — opción 'económica' (7 días, 6 noches, 2 personas):
- Pasajes micro semicama: $ 280.000  [$ 140.000 por persona (ida y vuelta) × 2 personas]
- Alojamiento hostel habitación doble: $ 210.000  [$ 35.000 por noche por habitación × 6 noches × 1 habitación (de 2 personas)]
- Comida: $ 126.000  [$ 9.000 por persona por día × 7 días × 2 personas]
- Actividades: $ 56.000  [$ 4.000 por persona por día × 7 días × 2 personas]
- excursión Isla Victoria: $ 113.100  [$ 56.550 × 2 personas]
TOTAL: $ 785.100 | PRESUPUESTO: $ 800.000
RESULTADO: ENTRA en el presupuesto. Sobran $ 14.900.
Costo por día: $ 112.157 | por persona por día: $ 56.079
Conversiones de moneda:
- excursión Isla Victoria: US$ 30,00 → $ 56.550 con dólar tarjeta $1.885 (pago con tarjeta; fuente: https://cotizaciones.mock-viajes.example/dolar-hoy)

**Con la opción económica el viaje ENTRA en el presupuesto.**

## Fuentes
- [web] https://cotizaciones.mock-viajes.example/dolar-hoy — cambio, buscado el 2026-11-10
- [web] https://micros.mock-viajes.example/mendoza-bariloche — pasajes, buscado el 2026-11-10
- [web] https://alojamiento.mock-viajes.example/bariloche-enero-precios — alojamiento, buscado el 2026-11-10
- [web] https://actividades.mock-viajes.example/bariloche-verano — comida, buscado el 2026-11-10

_Los precios son aproximados: salen de búsquedas web (o de la memoria local de búsquedas de los últimos días) y pueden cambiar. Confirmalos antes de reservar._

## 3. La misma pregunta en vivo (Claude + Tavily)

Solo corre si hay claves en el `.env`. La memoria es la real (`data/memoria_chroma/`): si la pregunta ya se hizo en los últimos 7 días, los datos salen de ahí.

In [5]:
if HAY_CLAVES:
    componentes_real = crear_componentes(modo_mock=False)
    grafo_real = construir_grafo(componentes_real)
    traza_real = await consultar(grafo_real, DEMO, escenario="notebook_demo", modo_mock=False)
    mostrar_recorrido(traza_real)
else:
    traza_real = None
    print("Sin claves: se salta la corrida en vivo (las trazas reales están en traces/).")

RUTA: supervisor → investigador → supervisor → analista → supervisor → sintesis

SUPERVISOR · pedido extraído: {'origen': 'Mendoza', 'destino': 'Bariloche', 'fecha_ida': '2027-01-10', 'fecha_vuelta': '2027-01-16', 'dias': 7, 'personas': 2, 'presupuesto': 800000.0, 'moneda': 'ARS', 'supuestos': ['Enero sin día exacto: ida el 10/01/2027 (próximo enero, ya que enero de 2026 ya pasó)', 'Vuelta calculada como ida + 6 días: 16/01/2027']}
SUPERVISOR · decisión 1 → investigador
   evaluación (rúbrica): Pregunta pide armar viaje y decir si alcanza el presupuesto (800.000 ARS, 2 personas, 7 días a Bariloche desde Mendoza). R1 requiere pasajes, alojamiento, comida/actividades y cotización del dólar. R2, R3, R4, R5, R6 aún no evaluables: no hay aportes todavía. Hace falta primero el investigador.
   temas pedidos: pasajes, alojamiento, comida, actividades, cambio
INVESTIGADOR · herramientas: buscar_en_memoria · llamadas: 5
   cobertura pasajes: 3 fuentes (3 memoria / 0 web) · con precios
   cobert

In [6]:
if traza_real is not None:
    display(Markdown(traza_real.respuesta_final))

## Itinerario

No armo un itinerario día por día: el analista no encontró actividades confirmadas con precio para enero de 2027. Lo que sí hay:

- **Teleférico Cerro Catedral** (con confitería giratoria): ARS 40.000 por adulto, excursión de 3 a 4 hs [https://bariloche.org/tarifas-excursiones-bariloche] (memoria) [mem-f2962de0fd34]
- **Excursiones de navegación y terrestres** (kayak, rafting, paseos en barco) disponibles en verano, pero sin precio en la fuente [https://www.tangol.com/es/excursiones/bariloche/d-5] (memoria) [mem-42bc3c7dcff9]

Como hay 7 días y pocas actividades confirmadas, sugiero dejar varios días libres para caminar, visitar el centro y disfrutar del lago. El clima esperado para enero no fue relevado.

## Presupuesto

Fechas: ida 10/01/2027, vuelta 16/01/2027 (7 días, 6 noches), 2 personas, presupuesto ARS 800.000. Todos los precios están en pesos argentinos (ARS), así que no hizo falta convertir moneda.

**Escenario principal (pasaje ida y vuelta por persona):**
- Pasajes en micro: ARS 270.200 (ARS 135.100 por persona × 2) [https://www.centraldepasajes.com.ar/cdp/pasajes-micro/mendoza/bariloche] (memoria) [mem-b6468acfdb03]
- Alojamiento Selva India Hotel Suite (precio "desde"): ARS 434.520 (ARS 72.420 por noche × 6 noches) [https://www.despegar.com.ar/hoteles/hl/901/i1/hoteles-en-san+carlos+de+bariloche] (memoria) [mem-f408466d015c]
- Comida: ARS 42.000 (ARS 3.000 por persona por día × 7 días × 2) [https://noticiasffha.com.ar/cuanto-cuesta-un-viaje-a-bariloche-y-como-planificarlo] (memoria) [mem-760d72e28e6f]
- **Total: ARS 746.720. Entra, y sobran ARS 53.280.**
- Costo por día: ARS 106.674 (ARS 53.337 por persona por día).

**Escenario de riesgo (pasaje por tramo, es decir ida y vuelta × 2):**
- Pasajes: ARS 540.400. Total: ARS 1.016.920.
- **No entra: faltan ARS 216.920.**

**Cotización de referencia (no se convirtió nada):** dólar oficial compra ARS 1.520 / venta ARS 1.570, con fecha 6/11/2026 [https://www.dolarito.ar] (memoria) [mem-13d552ddd1f0].

## Alternativas

Si el pasaje resulta ser por tramo, o si las comidas y excursiones reales cuestan más que el mínimo, el presupuesto no alcanza. Algunas formas de bajar costos:
- Reducir la estadía a 4 o 5 noches.
- Alojarse en una opción más barata que el hotel "desde" de referencia.
- Cocinar en el alojamiento en vez de comer en restaurantes (un desayuno cuesta ARS 9.000 [https://bariloche.org/cuanto_me_cuesta] (memoria) [mem-0019ce230e02]).
- Elegir actividades gratuitas como caminatas y costa del lago, y evitar el teleférico (ARS 40.000 por adulto).

## Lo que no pudimos averiguar

- **Confirmar el tipo de precio del pasaje:** la fuente no aclara si ARS 135.100 es por tramo o ida y vuelta. Es lo que más cambia el resultado.
- **Vuelos:** no se encontraron precios aéreos Mendoza–Bariloche.
- **Fechas:** los precios de alojamiento y pasajes son de noviembre y diciembre de 2026, no de enero de 2027.
- **Comida:** el gasto diario de ARS 3.000 por persona es el mínimo de la fuente y su unidad no está confirmada. Con comidas reales el costo sube bastante.
- **Excursiones de verano y travesías:** no tienen precio confiable.
- **Clima:** no se relevó para enero.
- **Búsqueda:** todos los datos salieron de memoria, sin búsqueda web en vivo.

Mi recomendación es confirmar el precio del pasaje y buscar precios de enero de 2027 antes de comprometer el dinero.

## Fuentes
- [memoria] mem-b6468acfdb03 → https://www.centraldepasajes.com.ar/cdp/pasajes-micro/mendoza/bariloche — pasajes, buscado el 2026-11-10
- [memoria] mem-f408466d015c → https://www.despegar.com.ar/hoteles/hl/901/i1/hoteles-en-san+carlos+de+bariloche — alojamiento, buscado el 2026-11-10
- [memoria] mem-0019ce230e02 → https://bariloche.org/cuanto_me_cuesta — comida, buscado el 2026-11-10
- [memoria] mem-760d72e28e6f → https://noticiasffha.com.ar/cuanto-cuesta-un-viaje-a-bariloche-y-como-planificarlo — comida, buscado el 2026-11-10
- [memoria] mem-42bc3c7dcff9 → https://www.tangol.com/es/excursiones/bariloche/d-5 — actividades, buscado el 2026-11-10
- [memoria] mem-f2962de0fd34 → https://bariloche.org/tarifas-excursiones-bariloche — actividades, buscado el 2026-11-10
- [memoria] mem-13d552ddd1f0 → https://www.dolarito.ar — cambio, buscado el 2026-11-10

_Los precios son aproximados: salen de búsquedas web (o de la memoria local de búsquedas de los últimos días) y pueden cambiar. Confirmalos antes de reservar._

## 4. Memoria: la segunda vez no se busca en la web

Se repite la pregunta de demo en mock, con la **misma** memoria ChromaDB del punto 2. El investigador consulta primero `buscar_en_memoria`; como los datos tienen menos de 7 días y son parecidos a la consulta, no llama a Tavily.

In [7]:
segunda = await consultar(grafo_mock, DEMO, escenario="notebook_memoria", modo_mock=True)
print("Primera vez →", traza_mock.texto_origenes(), "| búsquedas web:", traza_mock.metricas["busquedas_web"])
print("Segunda vez →", segunda.texto_origenes(), "| búsquedas web:", segunda.metricas["busquedas_web"])

Primera vez → 0 de memoria · 6 de la web | búsquedas web: 5
Segunda vez → 6 de memoria · 0 de la web | búsquedas web: 0


## 5. El freno de pasos (contra el "supervisor infinito")

Con `max_pasos=2`, la segunda decisión del supervisor es **FINALIZAR forzado por código** (sin consultar al LLM) y la síntesis avisa que la información es parcial.

In [8]:
freno = await consultar(construir_grafo(crear_componentes(modo_mock=True), max_pasos=2), DEMO,
                       escenario="notebook_freno", modo_mock=True, max_pasos=2)
mostrar_recorrido(freno)
print()
print(freno.respuesta_final.splitlines()[0])

RUTA: supervisor → investigador → supervisor → sintesis

SUPERVISOR · pedido extraído: {'origen': 'Mendoza', 'destino': 'Bariloche', 'fecha_ida': '2027-01-10', 'fecha_vuelta': '2027-01-16', 'dias': 7, 'personas': 2, 'presupuesto': 800000.0, 'moneda': 'ARS', 'supuestos': ['enero sin día exacto: se asume del 10/01/2027 al 16/01/2027']}
SUPERVISOR · decisión 1 → investigador
   evaluación (rúbrica): R1: todavía no hay datos. Empieza el investigador.
   temas pedidos: pasajes, alojamiento, clima, actividades, cambio
INVESTIGADOR · herramientas: buscar_en_memoria, buscar_en_web · llamadas: 10
   cobertura pasajes: 2 fuentes (0 memoria / 2 web) · con precios
   cobertura alojamiento: 1 fuentes (0 memoria / 1 web) · con precios
   cobertura clima: 1 fuentes (0 memoria / 1 web)
   cobertura actividades: 1 fuentes (0 memoria / 1 web) · con precios
   cobertura cambio: 1 fuentes (0 memoria / 1 web) · con precios
   chequeo por código: sin observaciones
SUPERVISOR · decisión 2 → FINALIZAR  (FORZA

## 6. Dónde está la evidencia

- `traces/` → una traza `.json` (estructurada) y una `.log` (legible) por corrida, con la ruta, cada decisión del supervisor con su evaluación, cada llamada a herramienta y las fuentes.
- `traces/resultados.md` → la tabla de los 8 escenarios en mock y en vivo (`python escenarios.py --tabla`).
- `tests/` → 128 tests con pytest que corren sin claves (`python -m pytest -q`).